In [1]:
import numpy as np
import numpy.random
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.utils import resample

In [2]:
intercept = 0.0
slope = 1.0
n = 11
x = np.linspace(0, 1.0, num=n)
sd_err = 0.1
seed = 1234
y = x + numpy.random.default_rng(seed=seed).normal(scale=sd_err, size=n)
x_pred = 0.75

I copy/pasted these values into R and did the linear regression there:

```
> predict(m, newdata=list(x=0.75), interval='confidence')
        fit       lwr       upr
1 0.7537439 0.6289261 0.8785618
```

The bootstrap estimate over linear regressions is close:


In [ ]:
def lm(x, y, x_pred=x_pred):
    return (
        LinearRegression().fit(X=x.reshape(-1, 1), y=y).predict(np.array([[x_pred]]))[0]
    )


def rf(x, y, **kwargs):
    return (
        RandomForestRegressor(**kwargs)
        .fit(X=x.reshape(-1, 1), y=y)
        .predict(np.array([[x_pred]]))[0]
    )


def bootstrap(f, x=x, y=y, **kwargs):
    bx, by = resample(x, y)
    return f(bx, by, **kwargs)


def summarize(est, alpha=0.05):
    print(
        f"{est.mean()} ({np.quantile(est, alpha / 2)} -- {np.quantile(est, 1 - alpha / 2)})"
    )


n_boot = 100
print("Bootstrap over linear regressions:")
summarize(np.array([bootstrap(lm) for _ in range(n_boot)]))
print("Bootstrap interval over individual trees:")
summarize(np.array([bootstrap(rf, n_estimators=1) for _ in range(n_boot)]))

n_trees = 1000
print(f"Quantiles of predictions from trees in a single forest of size {n_trees}:")
preds = np.array(
    [
        e.predict(np.array([[x_pred]]))[0]
        for e in RandomForestRegressor(n_estimators=n_trees)
        .fit(X=x.reshape(-1, 1), y=y)
        .estimators_
    ]
)
alpha = 0.05
print(
    np.mean(preds),
    "(",
    np.quantile(preds, alpha / 2),
    "--",
    np.quantile(preds, 1.0 - alpha / 2),
    ")",
)

n_trees = 100
print(f"Over forests of size {n_trees}:")
summarize(np.array([bootstrap(rf, n_estimators=n_trees) for _ in range(n_boot)]))

Bootstrap over linear regressions:
0.7590800273599796 (0.6691189538600754 -- 0.8483700322691561)
Bootstrap interval over individual trees:
0.7457392051535793 (0.4521176639335599 -- 0.934374458145268)
Quantiles of predictions from trees in a single forest of size 1000:
0.7286671051797036 ( 0.4521176639335599 -- 0.7945472974645861 )
Bootstrap over trees in a forest:
0.7286671051797036 ( 0.721103087971587 -- 0.7347363510833221 )
Over forests of size 100:
0.7092065339761559 (0.4643738802882021 -- 0.8472285654198348)


In [5]:
alpha = 0.05
for a in [alpha / 2, 0.5, 1 - alpha / 2]:
    pred = (
        GradientBoostingRegressor(loss="quantile", alpha=a, n_estimators=100)
        .fit(X=x.reshape(-1, 1), y=y)
        .predict(np.array([[x_pred]]))
    )
    print(f"{a=} {pred=}")

a=0.025 pred=array([0.10640468])
a=0.5 pred=array([0.64646172])
a=0.975 pred=array([0.93437474])
